In [ ]:
# ============================================================
# PROJECT 5: TITANIC SURVIVAL
# FULL LOGISTIC REGRESSION PIPELINE
# ============================================================

# ------------------------------------------------------------
# 1. IMPORT LIBRARIES
# ------------------------------------------------------------

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)

# Display settings
pd.set_option("display.max_columns", None)


# ------------------------------------------------------------
# 2. LOAD DATASET
# ------------------------------------------------------------

from google.colab import files

uploaded = files.upload()

# Make sure your file is named Titanic-Dataset.csv
df = pd.read_csv("Titanic-Dataset.csv")


# ------------------------------------------------------------
# 3. BASIC DATASET INFORMATION
# ------------------------------------------------------------

print("First 5 rows:")
display(df.head())

print("\nDataset Shape:")
print(df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nDataset Information:")
df.info()


# ------------------------------------------------------------
# 4. CHECK MISSING VALUES
# ------------------------------------------------------------

print("\nMissing Values:")
print(df.isnull().sum())


# ------------------------------------------------------------
# 5. CHECK DUPLICATES
# ------------------------------------------------------------

print("\nNumber of Duplicate Rows:")
print(df.duplicated().sum())


# ------------------------------------------------------------
# 6. EXPLORATORY DATA ANALYSIS
# ------------------------------------------------------------

print("\nSurvival Distribution:")
print(df["Survived"].value_counts())

plt.figure(figsize=(6, 4))

sns.countplot(
    data=df,
    x="Survived"
)

plt.title("Titanic Survival Distribution")
plt.xlabel("Survived")
plt.ylabel("Count")

plt.show()


# ------------------------------------------------------------
# 7. SURVIVAL BY GENDER
# ------------------------------------------------------------

plt.figure(figsize=(7, 5))

sns.countplot(
    data=df,
    x="Sex",
    hue="Survived"
)

plt.title("Survival by Gender")
plt.xlabel("Gender")
plt.ylabel("Count")

plt.show()


# ------------------------------------------------------------
# 8. SURVIVAL BY PASSENGER CLASS
# ------------------------------------------------------------

plt.figure(figsize=(7, 5))

sns.countplot(
    data=df,
    x="Pclass",
    hue="Survived"
)

plt.title("Survival by Passenger Class")
plt.xlabel("Passenger Class")
plt.ylabel("Count")

plt.show()


# ============================================================
# FEATURE ENGINEERING
# ============================================================

# ------------------------------------------------------------
# 9. CREATE FAMILY SIZE
# ------------------------------------------------------------

df["FamilySize"] = (
    df["SibSp"] +
    df["Parch"] +
    1
)

print("\nFamily Size:")
display(
    df[
        ["SibSp", "Parch", "FamilySize"]
    ].head()
)


# ------------------------------------------------------------
# 10. CREATE IS ALONE FEATURE
# ------------------------------------------------------------

df["IsAlone"] = (
    df["FamilySize"] == 1
).astype(int)

print("\nIsAlone:")
display(
    df[
        ["FamilySize", "IsAlone"]
    ].head(10)
)


# ------------------------------------------------------------
# 11. CREATE FAMILY SIZE CATEGORY
# ------------------------------------------------------------

df["FamilyCategory"] = pd.cut(
    df["FamilySize"],
    bins=[0, 1, 4, 7, 20],
    labels=[
        "Alone",
        "Small",
        "Medium",
        "Large"
    ]
)

print("\nFamily Category:")
display(
    df[
        ["FamilySize", "FamilyCategory"]
    ].head(10)
)


# ------------------------------------------------------------
# 12. CREATE TITLE FROM NAME
# ------------------------------------------------------------

df["Title"] = df["Name"].str.extract(
    r",\s*([^\.]+)\."
)

print("\nPassenger Titles:")
print(df["Title"].value_counts().head(15))


# ------------------------------------------------------------
# 13. GROUP RARE TITLES
# ------------------------------------------------------------

common_titles = [
    "Mr",
    "Miss",
    "Mrs",
    "Master"
]

df["Title"] = df["Title"].apply(
    lambda x: x if x in common_titles else "Rare"
)

print("\nGrouped Titles:")
print(df["Title"].value_counts())


# ------------------------------------------------------------
# 14. CREATE TARGET AND FEATURES
# ------------------------------------------------------------

y = df["Survived"]

X = df[
    [
        "Pclass",
        "Sex",
        "Age",
        "SibSp",
        "Parch",
        "Fare",
        "Embarked",
        "FamilySize",
        "IsAlone",
        "Title"
    ]
]

print("\nFeatures:")
display(X.head())

print("\nTarget:")
display(y.head())


# ------------------------------------------------------------
# 15. TRAIN-TEST SPLIT
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining Data Shape:")
print(X_train.shape)

print("\nTesting Data Shape:")
print(X_test.shape)


# ============================================================
# PREPROCESSING PIPELINE
# ============================================================

# ------------------------------------------------------------
# 16. DEFINE NUMERICAL FEATURES
# ------------------------------------------------------------

numeric_features = [
    "Pclass",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "FamilySize",
    "IsAlone"
]


# ------------------------------------------------------------
# 17. DEFINE CATEGORICAL FEATURES
# ------------------------------------------------------------

categorical_features = [
    "Sex",
    "Embarked",
    "Title"
]


# ------------------------------------------------------------
# 18. NUMERICAL PREPROCESSING
# ------------------------------------------------------------

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


# ------------------------------------------------------------
# 19. CATEGORICAL PREPROCESSING
# ------------------------------------------------------------

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)


# ------------------------------------------------------------
# 20. COMBINE PREPROCESSING
# ------------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)


# ============================================================
# LOGISTIC REGRESSION PIPELINE
# ============================================================

# ------------------------------------------------------------
# 21. CREATE COMPLETE PIPELINE
# ------------------------------------------------------------

logistic_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000
            )
        )
    ]
)


# ------------------------------------------------------------
# 22. TRAIN MODEL
# ------------------------------------------------------------

logistic_pipeline.fit(
    X_train,
    y_train
)

print("\nLogistic Regression model trained successfully.")


# ------------------------------------------------------------
# 23. MAKE PREDICTIONS
# ------------------------------------------------------------

y_pred = logistic_pipeline.predict(
    X_test
)

y_probability = logistic_pipeline.predict_proba(
    X_test
)[:, 1]


# ============================================================
# MODEL EVALUATION
# ============================================================

# ------------------------------------------------------------
# 24. ACCURACY
# ------------------------------------------------------------

accuracy = accuracy_score(
    y_test,
    y_pred
)

print("\nAccuracy:")
print(accuracy)


# ------------------------------------------------------------
# 25. PRECISION
# ------------------------------------------------------------

precision = precision_score(
    y_test,
    y_pred
)

print("\nPrecision:")
print(precision)


# ------------------------------------------------------------
# 26. RECALL
# ------------------------------------------------------------

recall = recall_score(
    y_test,
    y_pred
)

print("\nRecall:")
print(recall)


# ------------------------------------------------------------
# 27. F1 SCORE
# ------------------------------------------------------------

f1 = f1_score(
    y_test,
    y_pred
)

print("\nF1 Score:")
print(f1)


# ------------------------------------------------------------
# 28. ROC-AUC
# ------------------------------------------------------------

roc_auc = roc_auc_score(
    y_test,
    y_probability
)

print("\nROC-AUC Score:")
print(roc_auc)


# ------------------------------------------------------------
# 29. CLASSIFICATION REPORT
# ------------------------------------------------------------

print("\n================================================")
print("CLASSIFICATION REPORT")
print("================================================")

print(
    classification_report(
        y_test,
        y_pred
    )
)


# ------------------------------------------------------------
# 30. CONFUSION MATRIX
# ------------------------------------------------------------

cm = confusion_matrix(
    y_test,
    y_pred
)

print("\nConfusion Matrix:")
print(cm)


# ------------------------------------------------------------
# 31. CONFUSION MATRIX VISUALIZATION
# ------------------------------------------------------------

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=[
        "Did Not Survive",
        "Survived"
    ],
    yticklabels=[
        "Did Not Survive",
        "Survived"
    ]
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()


# ------------------------------------------------------------
# 32. ROC CURVE
# ------------------------------------------------------------

fpr, tpr, thresholds = roc_curve(
    y_test,
    y_probability
)

plt.figure(figsize=(8, 6))

plt.plot(
    fpr,
    tpr,
    label=f"Logistic Regression (AUC = {roc_auc:.3f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("ROC Curve")

plt.legend()

plt.show()


# ============================================================
# MODEL INTERPRETATION
# ============================================================

# ------------------------------------------------------------
# 33. GET FEATURE NAMES
# ------------------------------------------------------------

feature_names = (
    logistic_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

print("\nNumber of Processed Features:")
print(len(feature_names))


# ------------------------------------------------------------
# 34. GET LOGISTIC REGRESSION COEFFICIENTS
# ------------------------------------------------------------

coefficients = (
    logistic_pipeline
    .named_steps["classifier"]
    .coef_[0]
)

coefficient_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})

coefficient_df[
    "AbsoluteCoefficient"
] = coefficient_df[
    "Coefficient"
].abs()

coefficient_df = coefficient_df.sort_values(
    by="AbsoluteCoefficient",
    ascending=False
)

print("\nLogistic Regression Coefficients:")
display(coefficient_df)


# ------------------------------------------------------------
# 35. COEFFICIENT VISUALIZATION
# ------------------------------------------------------------

top_coefficients = coefficient_df.head(15)

plt.figure(figsize=(10, 7))

sns.barplot(
    data=top_coefficients,
    x="Coefficient",
    y="Feature"
)

plt.title(
    "Top Logistic Regression Feature Coefficients"
)

plt.xlabel("Coefficient")
plt.ylabel("Feature")

plt.show()


# ============================================================
# SAMPLE PREDICTIONS
# ============================================================

# ------------------------------------------------------------
# 36. COMPARE ACTUAL AND PREDICTED VALUES
# ------------------------------------------------------------

comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred,
    "Survival_Probability": y_probability
})

print("\nActual vs Predicted:")
display(comparison.head(20))


# ------------------------------------------------------------
# 37. PREDICT A NEW PASSENGER
# ------------------------------------------------------------

new_passenger = pd.DataFrame({
    "Pclass": [3],
    "Sex": ["male"],
    "Age": [25],
    "SibSp": [0],
    "Parch": [0],
    "Fare": [10],
    "Embarked": ["S"],
    "FamilySize": [1],
    "IsAlone": [1],
    "Title": ["Mr"]
})

new_prediction = logistic_pipeline.predict(
    new_passenger
)

new_probability = logistic_pipeline.predict_proba(
    new_passenger
)[:, 1]

print("\n================================================")
print("NEW PASSENGER PREDICTION")
print("================================================")

if new_prediction[0] == 1:
    print("Prediction: SURVIVED")
else:
    print("Prediction: DID NOT SURVIVE")

print(
    "Probability of Survival:",
    round(new_probability[0] * 100, 2),
    "%"
)


# ============================================================
# 38. FINAL MODEL SUMMARY
# ============================================================

results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "ROC-AUC"
    ],

    "Score": [
        accuracy,
        precision,
        recall,
        f1,
        roc_auc
    ]
})

print("\n================================================")
print("FINAL MODEL PERFORMANCE")
print("================================================")

display(results)


# ============================================================
# 39. PROJECT CONCLUSION
# ============================================================

print("""
============================================================
PROJECT 5 CONCLUSION
============================================================

Problem:
Predict whether a Titanic passenger survived.

Target Variable:
Survived

Feature Engineering:
1. FamilySize
2. IsAlone
3. Title

Preprocessing:
1. Missing-value imputation
2. StandardScaler for numerical variables
3. OneHotEncoder for categorical variables

Model:
Logistic Regression

Evaluation Metrics:
1. Accuracy
2. Precision
3. Recall
4. F1 Score
5. ROC-AUC
6. Confusion Matrix
7. ROC Curve

The complete preprocessing and machine-learning workflow
was implemented using a Scikit-learn Pipeline.

This prevents data preprocessing steps from being performed
separately and helps maintain a consistent ML workflow.

============================================================
PROJECT 5 COMPLETE
============================================================
""")